In [1]:
# Core Python libraries
import random
import pandas as pd

# PyTorch libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Hugging Face Transformers & PEFT (LoRA)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
def generate_dataset(n_samples=200):
    """
    Generates a simple synthetic dataset for sentiment analysis.
    Label convention:
    1 -> Positive sentiment
    0 -> Negative sentiment
    """

    # Positive and negative sentence templates
    positive_templates = [
        "I love this product",
        "This is an amazing experience",
        "Absolutely fantastic service",
        "I am very happy with the results",
        "This works perfectly"
    ]

    negative_templates = [
        "I hate this product",
        "This is a terrible experience",
        "Absolutely horrible service",
        "I am very disappointed",
        "This does not work at all"
    ]

    data = []

    # Create balanced positive and negative samples
    for _ in range(n_samples // 2):
        data.append((random.choice(positive_templates), 1))
        data.append((random.choice(negative_templates), 0))

    # Shuffle data to avoid ordering bias
    random.shuffle(data)

    # Convert to Pandas DataFrame
    return pd.DataFrame(data, columns=["text", "label"])


# Generate dataset
df = generate_dataset(200)
print(df.head())


                            text  label
0  This is an amazing experience      1
1  This is a terrible experience      0
2  This is a terrible experience      0
3   Absolutely fantastic service      1
4            I love this product      1


In [3]:
# Split dataset while preserving class distribution
train_df, val_df = train_test_split(
    df,
    test_size=0.2,          # 80% train, 20% validation
    random_state=42,
    stratify=df["label"]    # Maintain class balance
)


In [4]:
# Load BERT tokenizer (converts text into token IDs)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
class SentimentDataset(Dataset):
    """
    Custom PyTorch Dataset class for sentiment analysis.
    Converts text into tokenized inputs and provides labels.
    """

    def __init__(self, texts, labels):
        # Tokenize all input texts
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,    # Truncate long sequences
            padding=True,       # Pad shorter sequences
            max_length=128
        )
        self.labels = labels.tolist()

    def __len__(self):
        # Returns total number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieves one sample at index idx
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


# Create dataset objects
train_dataset = SentimentDataset(train_df["text"], train_df["label"])
val_dataset   = SentimentDataset(val_df["text"], val_df["label"])


In [6]:
# Load pre-trained BERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2   # Binary sentiment classification
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# Configure LoRA adapters
lora_config = LoraConfig(
    r=8,                      # Rank of low-rank matrices
    lora_alpha=32,            # Scaling factor
    target_modules=["query", "value"],  # Apply LoRA to attention layers
    lora_dropout=0.1,         # Dropout for regularization
    bias="none",
    task_type="SEQ_CLS"       # Sequence classification task
)

# Wrap the BERT model with LoRA adapters
model = get_peft_model(model, lora_config)

# Print number of trainable vs total parameters
model.print_trainable_parameters()


trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [8]:
# Create PyTorch DataLoaders for batching
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)


In [9]:
# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# AdamW optimizer (recommended for Transformers)
optimizer = AdamW(
    model.parameters(),
    lr=2e-4,           # Higher LR is suitable for LoRA
    weight_decay=0.01
)

# Cross-entropy loss for classification
criterion = nn.CrossEntropyLoss()


In [10]:
def train_epoch(model, dataloader):
    """
    Trains the model for one epoch.
    """
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass
        outputs = model(**batch)

        # Compute loss
        loss = outputs.loss

        # Backpropagation
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [11]:
def evaluate(model, dataloader):
    """
    Evaluates the model on validation data.
    Returns accuracy score.
    """
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(batch["labels"].cpu().numpy())

    return accuracy_score(true_labels, predictions)


In [12]:
epochs = 5

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    val_accuracy = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print("-" * 50)


Epoch 1/5
Training Loss: 0.6829
Validation Accuracy: 0.8750
--------------------------------------------------
Epoch 2/5
Training Loss: 0.6144
Validation Accuracy: 0.9250
--------------------------------------------------
Epoch 3/5
Training Loss: 0.3517
Validation Accuracy: 1.0000
--------------------------------------------------
Epoch 4/5
Training Loss: 0.0643
Validation Accuracy: 1.0000
--------------------------------------------------
Epoch 5/5
Training Loss: 0.0116
Validation Accuracy: 1.0000
--------------------------------------------------


In [13]:
# Save LoRA-adapted model and tokenizer
model.save_pretrained("bert-lora-sentiment")
tokenizer.save_pretrained("bert-lora-sentiment")


('bert-lora-sentiment/tokenizer_config.json',
 'bert-lora-sentiment/special_tokens_map.json',
 'bert-lora-sentiment/vocab.txt',
 'bert-lora-sentiment/added_tokens.json',
 'bert-lora-sentiment/tokenizer.json')

In [14]:
from transformers import pipeline

# Create inference pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="bert-lora-sentiment",
    tokenizer="bert-lora-sentiment"
)

print(sentiment_pipeline("This is an excellent model!"))
print(sentiment_pipeline("I really hate this approach."))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.9974033236503601}]
[{'label': 'LABEL_0', 'score': 0.9970107078552246}]
